# Optimised scratch CNN - dedicated training notebook
## BRISC 2025 | Catarina Bota

Best-effort from-scratch CNN, **symmetric to the optimised EfficientNet track**, so the "transfer learning is decisive" claim is defensible.

**Non-destructive:** does NOT touch the canonical `modelo_brisc.h5`. Outputs go to `scratch_cnn_optimized/`.

Improvements over the 3-block canonical CNN: BatchNorm, a 4th conv block, Global Average Pooling (fewer params), dropout + L2, ReduceLROnPlateau schedule, up to 80 epochs, stronger augmentation. Same 128px / rescale-255 input, so the gain is attributable to architecture + training only.

**How to run:** confirm the kernel (cell below), then run top to bottom. Training is ~15–25 min.

## 0. Environment check

In [5]:
import os, sys, json, subprocess
PROJ = '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui'
BASE = '/root/brisc/data/brisc2025_clean/classification_task'   # audited partition
OUT  = f'{PROJ}/scratch_cnn_optimized'

print('kernel python :', sys.executable)
assert 'brisc' in sys.executable, 'WRONG KERNEL -> Kernel > Change Kernel > /root/brisc/.venv'
print('worker exists :', os.path.exists(f'{PROJ}/train_scratch_cnn_optimized.py'))
print('data exists   :', os.path.isdir(BASE))
print('output dir    :', OUT)

kernel python : /root/brisc/.venv/bin/python3
worker exists : True
data exists   : True
output dir    : /mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/scratch_cnn_optimized


## 1. Train (audited)  —  ~15–25 min

You will see the training run epoch by epoch. The red `cuFFT/cuDNN registered` lines are harmless warnings. EarlyStopping may stop before 80 epochs.

In [6]:
!"{sys.executable}" "{PROJ}/train_scratch_cnn_optimized.py" audited "{BASE}" "{OUT}"

E0000 00:00:1783544468.614367    3496 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783544468.636999    3496 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783544468.802209    3496 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783544468.802284    3496 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783544468.802288    3496 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783544468.802290    3496 computation_placer.cc:177] computation placer already registered. Please check linka

## 2. Result + comparison

In [7]:
m = json.load(open(f'{OUT}/audited/metrics.json'))
print(f"Optimised scratch CNN   : {m['accuracy']:.2f}%  "
      f"(F1 {m['f1_macro']:.2f}, AUC {m['auc_ovr']:.2f}, "
      f"CI95 [{m['ci95_lo']}, {m['ci95_hi']}], {m['params']:,} params, {m['epochs']} ep)")
print('\n--- context (audited test set) ---')
print('Canonical scratch CNN   : 93.81%')
print('VGG16 transfer softmax  : 97.94%')
print('Best hybrid (VGG16+LGBM): 98.67%')
print('EfficientNetB1 optimised: 98.82%')
print(f"\nGain vs canonical scratch: {m['accuracy']-93.81:+.2f} pp")

Optimised scratch CNN   : 91.89%  (F1 92.18, AUC 98.66, CI95 [89.82, 93.81], 648,996 params, 80 ep)

--- context (audited test set) ---
Canonical scratch CNN   : 93.81%
VGG16 transfer softmax  : 97.94%
Best hybrid (VGG16+LGBM): 98.67%
EfficientNetB1 optimised: 98.82%

Gain vs canonical scratch: -1.92 pp


## 3. Regenerate learning curves

Adds the **"Scratch CNN opt."** panel to `fig_training_history` / `fig_training_loss`.

In [8]:
subprocess.run([sys.executable, f'{PROJ}/build_training_history.py'], check=True)
print('\nLearning-curve figures regenerated.')

saved fig_training_history
saved fig_training_loss
done -> figures in figuras_artigo/ and latex_artigo/figures/

Learning-curve figures regenerated.


## Notes

- **Non-destructive:** `modelo_brisc.h5` (canonical scratch CNN) is untouched.
- **Deterministic** (seed 42) → re-running reproduces the same result.
- After it finishes, wire the accuracy into the supplementary tables.
- For the **unaudited** number too, copy cell 1 and change `audited` → `unaudited` and `BASE` → `/root/brisc/data_unaudited/classification_task`.